In [10]:
import math
import random
import pandas as pd
import numpy as np   
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

ITERATIONS = 100

rice = "../res/rice.csv"
parkinsons = "../res/parkinsons.csv"
credits = "../res/credit_approval.csv"

def stratified_k_fold(df, k=5, y="label"):
    folds = [[] for _ in range(k)]

    classes = df[y].unique()
    class_subsets = {}

    for c in classes:
        subset = df[df[y] == c].sample(frac=1).reset_index(drop=True)
        class_subsets[c] = subset

    for c in classes:
        subset = class_subsets[c]
        for i in range(len(subset)):
            folds[i % k].append(subset.iloc[i])

    folds = [pd.DataFrame(fold).reset_index(drop=True) for fold in folds]

    return folds


def confusion_matrix(preds, labels, positive_class):
    TP = FP = TN = FN = 0

    for i in range(len(labels)):
        if labels[i] == positive_class and preds[i] == positive_class:
            TP += 1
        elif labels[i] != positive_class and preds[i] == positive_class:
            FP += 1
        elif labels[i] != positive_class and preds[i] != positive_class:
            TN += 1
        else:
            FN += 1

    return TP, FP, TN, FN

def accuracy(TP, FP, TN, FN):
    return (TP + TN) / (TP + FP + TN + FN + 1e-12)

def precision(TP, FP):
    return TP / (TP + FP + 1e-12)

def recall(TP, FN):
    return TP / (TP + FN + 1e-12)

def f1(TP, FP, FN):
    p = precision(TP, FP)
    r = recall(TP, FN)
    return 2 * p * r / (p + r + 1e-12)


def preprocess_dataset(df, target_col):
    X_raw = df.drop(target_col, axis=1).copy()
    Y = df[target_col].values

    # separate categories and numbers
    cat_cols = [c for c in X_raw.columns if "_cat" in c or X_raw[c].dtype == 'object']
    num_cols = [c for c in X_raw.columns if "_num" in c or X_raw[c].dtype in ['int64', 'float64', 'float32']]

    # factorize cat 
    X_cat_encoded = np.zeros((X_raw.shape[0], len(cat_cols)))
    for i, col in enumerate(cat_cols):
        # pd.factorize returns (labels, uniques)
        X_cat_encoded[:, i] = pd.factorize(X_raw[col])[0]

    # normalize for num
    X_num_scaled = np.empty((X_raw.shape[0], 0)) # default if no num_cols exist
    if num_cols:
        scaler = MinMaxScaler()
        X_num_scaled = scaler.fit_transform(X_raw[num_cols])

    # hstack merges the columns into a single feature matrix
    if len(cat_cols) > 0 and len(num_cols) > 0:
        X = np.hstack((X_num_scaled, X_cat_encoded))
    elif len(cat_cols) > 0:
        X = X_cat_encoded
    else:
        X = X_num_scaled

    return X, Y

class NeuralNetwork:
    def __init__(self, architecture, alpha=0.1, lambd=0.1):
        self.alpha = alpha
        self.lambd = lambd
        self.weights = []
        for i in range(len(architecture) - 1):
            # weight initialized to small random values
            w = np.random.uniform(-1, 1, (architecture[i+1], architecture[i] + 1))
            self.weights.append(w)

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def sigmoid_deriv(self, a):
        return a * (1 - a)

    def forward(self, x):
        # initial activation = the input with bias
        activations = [np.vstack(([1], x.reshape(-1, 1)))]
        for i, W in enumerate(self.weights):
            z = W @ activations[-1]
            a_next = self.sigmoid(z)
            # add bias for hidden layers, but not the output layer
            if i < len(self.weights) - 1:
                a_next = np.vstack(([1], a_next))
            activations.append(a_next)
        return activations

    def train(self, X, Y, iterations=ITERATIONS):
        m = len(X)
        for _ in range(iterations):
            grad_sums = [np.zeros_like(W) for W in self.weights]
            
            for i in range(m):
                acts = self.forward(X[i])
                deltas = [None] * len(self.weights)

                # output layer delta
                deltas[-1] = acts[-1] - Y[i].reshape(-1, 1)
                
                # hidden layer deltas (backpropagate)
                for j in range(len(self.weights) - 2, -1, -1):
                    deltas[j] = (self.weights[j+1][:, 1:].T @ deltas[j+1]) * self.sigmoid_deriv(acts[j+1][1:])
                
                for j in range(len(self.weights)):
                    grad_sums[j] += deltas[j] @ acts[j].T

            # update weights with regularization
            for j in range(len(self.weights)):
                # setting bias weight penalty to 0
                reg_matrix = (self.lambd / m) * self.weights[j]
                reg_matrix[:, 0] = 0
                
                grad = (grad_sums[j] / m) + reg_matrix
                self.weights[j] -= self.alpha * grad

    def predict(self, X, threshold=0.5):
        preds = [1 if self.forward(x)[-1][0, 0] >= threshold else 0 for x in X]
        return np.array(preds)

    def compute_cost(self, X, Y):
        m = len(X)
        total_j = 0
        for i in range(m):
            h = self.forward(X[i])[-1]
            total_j += np.sum(-Y[i] * np.log(h + 1e-12) - (1 - Y[i]) * np.log(1 - h + 1e-12))
        
        reg_term = sum(np.sum(W[:, 1:]**2) for W in self.weights)
        return (total_j / m) + (self.lambd / (2 * m)) * reg_term


def run_experiments(df, architectures, lambdas, alphas, target):
    X, Y = preprocess_dataset(df, target)
    df_proc = pd.DataFrame(X)
    df_proc[target] = Y
    
    results = []
    
    for arch in architectures:
        for lam in lambdas:
            for alpha in alphas:
                print(f"Running: Arch={arch}, Lambda={lam}, Alpha={alpha}")
                
                folds = stratified_k_fold(df_proc, k=5, y=target)
                fold_acc, fold_f1 = [], []
                
                for i in range(5):
                    test = folds[i]
                    train = pd.concat([folds[j] for j in range(5) if i != j])
                    
                    X_train, Y_train = train.drop(target, axis=1).values, train[target].values
                    X_test, Y_test = test.drop(target, axis=1).values, test[target].values
                    
                    nn = NeuralNetwork(architecture=arch, alpha=alpha, lambd=lam)
                    nn.train(X_train, Y_train, iterations=ITERATIONS)
                    
                    preds = nn.predict(X_test)
                    tp, fp, tn, fn = confusion_matrix(preds, Y_test, 1)
                    fold_acc.append(accuracy(tp, fp, tn, fn))
                    fold_f1.append(f1(tp, fp, fn))
                
                results.append({
                    "Architecture": str(arch),
                    "Lambda": lam,
                    "Alpha": alpha,
                    "Avg_Accuracy": np.mean(fold_acc),
                    "Avg_F1": np.mean(fold_f1)
                })
    
    return pd.DataFrame(results)

def plot_learning_curve(X, Y, best_arch, alpha, lambd):

    X_train_full, X_test, Y_train_full, Y_test = train_test_split(
        X, Y, test_size=0.2, stratify=Y
    )
    
    sizes = np.linspace(10, len(X_train_full), 10, dtype=int)
    costs = []
    
    for s in sizes:
        nn = NeuralNetwork(architecture=[X.shape[1]] + best_arch + [1], alpha=alpha, lambd=lambd)
        nn.train(X_train_full[:s], Y_train_full[:s], iterations=ITERATIONS)
        
        # calculate cost J on test set
        current_cost = nn.compute_cost(X_test, Y_test)
        costs.append(current_cost)
        
    plt.figure(figsize=(8, 5))
    plt.plot(sizes, costs, 'b-o', label='Test Cost (J)')
    plt.title(f"Learning Curve (Best Arch: {best_arch}, Alpha: {alpha})")
    plt.xlabel("Number of Training Samples")
    plt.ylabel("Cost J on Test Set")
    plt.grid(True)
    plt.legend()
    plt.show()


In [ ]:
rice_df = pd.read_csv(rice)
if rice_df['label'].dtype == 'object':
    rice_df['label'] = pd.factorize(rice_df['label'])[0]

X_rice_proc, Y_rice_proc = preprocess_dataset(rice_df, target_col="label")
input_dim = X_rice_proc.shape[1]

print(f"Detected Input Dimension: {input_dim}")

rice_archs = [
    [input_dim, 2, 1],
    [input_dim, 8, 1],
    [input_dim, 16, 8, 4, 1],
    [input_dim, 8, 8, 8, 1],
    [input_dim, 4, 32, 1],
    [input_dim, 4, 4, 1]
]

rice_lambdas = [0.00001, 0.01, 1.0]
rice_alphas = [1.0]
rice_experimental_results = run_experiments(
    rice_df, 
    architectures=rice_archs, 
    lambdas=rice_lambdas, 
    alphas=rice_alphas,
    target="label"
)

print("\n--- Rice Experimental Results ---")
print(rice_experimental_results.sort_values(by='Avg_Accuracy', ascending=False).to_string(index=False))

Detected Input Dimension: 7
Running: Arch=[7, 2, 1], Lambda=1e-05, Alpha=1.0
Running: Arch=[7, 2, 1], Lambda=0.01, Alpha=1.0
Running: Arch=[7, 2, 1], Lambda=1.0, Alpha=1.0
Running: Arch=[7, 8, 1], Lambda=1e-05, Alpha=1.0
Running: Arch=[7, 8, 1], Lambda=0.01, Alpha=1.0
Running: Arch=[7, 8, 1], Lambda=1.0, Alpha=1.0
Running: Arch=[7, 16, 8, 4, 1], Lambda=1e-05, Alpha=1.0
Running: Arch=[7, 16, 8, 4, 1], Lambda=0.01, Alpha=1.0
Running: Arch=[7, 16, 8, 4, 1], Lambda=1.0, Alpha=1.0
Running: Arch=[7, 8, 8, 8, 1], Lambda=1e-05, Alpha=1.0
Running: Arch=[7, 8, 8, 8, 1], Lambda=0.01, Alpha=1.0
Running: Arch=[7, 8, 8, 8, 1], Lambda=1.0, Alpha=1.0
Running: Arch=[7, 4, 32, 1], Lambda=1e-05, Alpha=1.0
Running: Arch=[7, 4, 32, 1], Lambda=0.01, Alpha=1.0
Running: Arch=[7, 4, 32, 1], Lambda=1.0, Alpha=1.0
Running: Arch=[7, 4, 4, 1], Lambda=1e-05, Alpha=1.0
Running: Arch=[7, 4, 4, 1], Lambda=0.01, Alpha=1.0
Running: Arch=[7, 4, 4, 1], Lambda=1.0, Alpha=1.0

--- Raisins Experimental Results ---
    Archit

In [12]:
parkinsons_df = pd.read_csv(parkinsons)
if parkinsons_df['Diagnosis'].dtype == 'object':
    parkinsons_df['Diagnosis'] = pd.factorize(parkinsons_df['Diagnosis'])[0]

X_parkinsons_proc, Y_parkinsons_proc = preprocess_dataset(parkinsons_df, target_col="Diagnosis")
input_dim = X_parkinsons_proc.shape[1]

print(f"Detected Input Dimension: {input_dim}")

parkinsons_archs = [
    [input_dim, 2, 1],
    [input_dim, 8, 1],
    [input_dim, 16, 8, 4, 1],
    [input_dim, 8, 8, 8, 1],
    [input_dim, 4, 32, 1],
    [input_dim, 4, 4, 1]
]

parkinsons_lambdas = [0.000001, 0.001, 1.0]
parkinsons_alphas = [1.0]
parkinsons_experimental_results = run_experiments(
    parkinsons_df, 
    architectures=parkinsons_archs, 
    lambdas=parkinsons_lambdas, 
    alphas=parkinsons_alphas,
    target="Diagnosis"
)

print("\n--- Parkinsons Experimental Results ---")
print(parkinsons_experimental_results.sort_values(by='Avg_Accuracy', ascending=False).to_string(index=False))

Detected Input Dimension: 22
Running: Arch=[22, 2, 1], Lambda=1e-06, Alpha=1.0
Running: Arch=[22, 2, 1], Lambda=0.001, Alpha=1.0
Running: Arch=[22, 2, 1], Lambda=1.0, Alpha=1.0
Running: Arch=[22, 8, 1], Lambda=1e-06, Alpha=1.0
Running: Arch=[22, 8, 1], Lambda=0.001, Alpha=1.0
Running: Arch=[22, 8, 1], Lambda=1.0, Alpha=1.0
Running: Arch=[22, 16, 8, 4, 1], Lambda=1e-06, Alpha=1.0
Running: Arch=[22, 16, 8, 4, 1], Lambda=0.001, Alpha=1.0
Running: Arch=[22, 16, 8, 4, 1], Lambda=1.0, Alpha=1.0
Running: Arch=[22, 8, 8, 8, 1], Lambda=1e-06, Alpha=1.0
Running: Arch=[22, 8, 8, 8, 1], Lambda=0.001, Alpha=1.0
Running: Arch=[22, 8, 8, 8, 1], Lambda=1.0, Alpha=1.0
Running: Arch=[22, 4, 32, 1], Lambda=1e-06, Alpha=1.0
Running: Arch=[22, 4, 32, 1], Lambda=0.001, Alpha=1.0
Running: Arch=[22, 4, 32, 1], Lambda=1.0, Alpha=1.0
Running: Arch=[22, 4, 4, 1], Lambda=1e-06, Alpha=1.0
Running: Arch=[22, 4, 4, 1], Lambda=0.001, Alpha=1.0
Running: Arch=[22, 4, 4, 1], Lambda=1.0, Alpha=1.0

--- Parkinsons Experim

In [7]:
credit_df = pd.read_csv(credits)
if credit_df['label'].dtype == 'object':
    credit_df['label'] = pd.factorize(credit_df['label'])[0]

X_credit_proc, Y_credit_proc = preprocess_dataset(credit_df, target_col="label")
input_dim = X_credit_proc.shape[1]

print(f"Detected Input Dimension: {input_dim}")

credit_archs = [
    [input_dim, 2, 1],
    [input_dim, 8, 1],
    [input_dim, 16, 8, 4, 1],
    [input_dim, 8, 8, 8, 1],
    [input_dim, 4, 32, 1],
    [input_dim, 4, 4, 1]
]

credit_lambdas = [0.00001, 0.01, 1.0]
credit_alphas = [1.0]
credit_experimental_results = run_experiments(
    credit_df, 
    architectures=credit_archs, 
    lambdas=credit_lambdas, 
    alphas=credit_alphas,
    target="label"
)

print("\n--- Credit Experimental Results ---")
print(credit_experimental_results.sort_values(by='Avg_Accuracy', ascending=False).to_string(index=False))

Detected Input Dimension: 16
Running: Arch=[16, 2, 1], Lambda=1e-05, Alpha=1.0
Running: Arch=[16, 2, 1], Lambda=0.01, Alpha=1.0
Running: Arch=[16, 2, 1], Lambda=1.0, Alpha=1.0
Running: Arch=[16, 8, 1], Lambda=1e-05, Alpha=1.0
Running: Arch=[16, 8, 1], Lambda=0.01, Alpha=1.0
Running: Arch=[16, 8, 1], Lambda=1.0, Alpha=1.0
Running: Arch=[16, 16, 8, 4, 1], Lambda=1e-05, Alpha=1.0
Running: Arch=[16, 16, 8, 4, 1], Lambda=0.01, Alpha=1.0
Running: Arch=[16, 16, 8, 4, 1], Lambda=1.0, Alpha=1.0
Running: Arch=[16, 8, 8, 8, 1], Lambda=1e-05, Alpha=1.0
Running: Arch=[16, 8, 8, 8, 1], Lambda=0.01, Alpha=1.0
Running: Arch=[16, 8, 8, 8, 1], Lambda=1.0, Alpha=1.0
Running: Arch=[16, 4, 32, 1], Lambda=1e-05, Alpha=1.0
Running: Arch=[16, 4, 32, 1], Lambda=0.01, Alpha=1.0
Running: Arch=[16, 4, 32, 1], Lambda=1.0, Alpha=1.0
Running: Arch=[16, 4, 4, 1], Lambda=1e-05, Alpha=1.0
Running: Arch=[16, 4, 4, 1], Lambda=0.01, Alpha=1.0
Running: Arch=[16, 4, 4, 1], Lambda=1.0, Alpha=1.0

--- Credit Experimental Resu